In [1]:
# M24CA1L306 Data Science Lab - Lab Exercise #06
# Student: Abhinav | MAC25MCA-2002 | S3 MCA (2025-27)
# Dataset: iris.csv
# Task: kNN using sklearn - KNeighborsClassifier and KNeighborsRegressor

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, mean_squared_error, r2_score
)

In [2]:
# ─────────────────────────────────────────────
# Load and prepare data
# ─────────────────────────────────────────────
df = pd.read_csv("Iris.csv")
df.drop(columns=["Id"], inplace=True)

X = df[["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]].values

# Encode species labels to integers (0, 1, 2)
le = LabelEncoder()
y_class = le.fit_transform(df["Species"])       # for classification
y_reg   = df["PetalLengthCm"].values            # for regression (predict petal length)

# Scale features - kNN is distance-based, so scaling matters
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [3]:
# ─────────────────────────────────────────────
# Train/Test Split
# ─────────────────────────────────────────────
X_train, X_test, y_train_cls, y_test_cls = train_test_split(
    X_scaled, y_class, test_size=0.2, random_state=42, stratify=y_class
)
_, _, y_train_reg, y_test_reg = train_test_split(
    X_scaled, y_reg, test_size=0.2, random_state=42
)

In [4]:
# ─────────────────────────────────────────────
# KNeighborsClassifier
# ─────────────────────────────────────────────
print("=" * 50)
print("KNeighborsClassifier (k=5)")
print("=" * 50)

knn_clf = KNeighborsClassifier(n_neighbors=5, metric="euclidean")
knn_clf.fit(X_train, y_train_cls)

y_pred_cls = knn_clf.predict(X_test)

print(f"Accuracy       : {accuracy_score(y_test_cls, y_pred_cls):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_cls, y_pred_cls, target_names=le.classes_))
print("Confusion Matrix:")
print(confusion_matrix(y_test_cls, y_pred_cls))

# Predict a new instance
new_instance = np.array([[5.1, 3.5, 1.4, 0.2]])           # raw values
new_scaled   = scaler.transform(new_instance)
pred_label   = knn_clf.predict(new_scaled)
pred_proba   = knn_clf.predict_proba(new_scaled)

print("\nNew instance (raw)  :", new_instance[0])
print("Predicted class     :", le.inverse_transform(pred_label)[0])
print("Class probabilities :", dict(zip(le.classes_, pred_proba[0].round(3))))

KNeighborsClassifier (k=5)
Accuracy       : 0.9333

Classification Report:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       0.83      1.00      0.91        10
 Iris-virginica       1.00      0.80      0.89        10

       accuracy                           0.93        30
      macro avg       0.94      0.93      0.93        30
   weighted avg       0.94      0.93      0.93        30

Confusion Matrix:
[[10  0  0]
 [ 0 10  0]
 [ 0  2  8]]

New instance (raw)  : [5.1 3.5 1.4 0.2]
Predicted class     : Iris-setosa
Class probabilities : {'Iris-setosa': np.float64(1.0), 'Iris-versicolor': np.float64(0.0), 'Iris-virginica': np.float64(0.0)}


In [5]:
# ─────────────────────────────────────────────
# KNeighborsRegressor
# Predicts PetalLengthCm from sepal + petal width features
# ─────────────────────────────────────────────
print("\n" + "=" * 50)
print("KNeighborsRegressor (k=5) - Predicting Petal Length")
print("=" * 50)

# Use SepalLengthCm, SepalWidthCm, PetalWidthCm as regressors
X_reg = df[["SepalLengthCm", "SepalWidthCm", "PetalWidthCm"]].values
X_reg_scaled = StandardScaler().fit_transform(X_reg)

X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(
    X_reg_scaled, y_reg, test_size=0.2, random_state=42
)

knn_reg = KNeighborsRegressor(n_neighbors=5, metric="euclidean")
knn_reg.fit(X_r_train, y_r_train)

y_pred_reg = knn_reg.predict(X_r_test)

mse  = mean_squared_error(y_r_test, y_pred_reg)
rmse = np.sqrt(mse)
r2   = r2_score(y_r_test, y_pred_reg)

print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

# Predict petal length for a new instance
new_reg_raw    = np.array([[5.0, 3.4, 0.3]])
new_reg_scaled = StandardScaler().fit_transform(
    np.vstack([X_reg, new_reg_raw])
)[-1].reshape(1, -1)

# Simpler: reuse fitted scaler
reg_scaler = StandardScaler().fit(X_reg)
new_reg_scaled = reg_scaler.transform(new_reg_raw)

predicted_petal_len = knn_reg.predict(new_reg_scaled)
print(f"\nNew instance (SepalLen, SepalWid, PetalWid): {new_reg_raw[0]}")
print(f"Predicted Petal Length : {predicted_petal_len[0]:.4f} cm")



KNeighborsRegressor (k=5) - Predicting Petal Length
MSE  : 0.1043
RMSE : 0.3230
R²   : 0.9682

New instance (SepalLen, SepalWid, PetalWid): [5.  3.4 0.3]
Predicted Petal Length : 1.4200 cm


In [6]:

# ─────────────────────────────────────────────
# Effect of k on classifier accuracy
# ─────────────────────────────────────────────
print("\n" + "=" * 50)
print("Accuracy vs k (KNeighborsClassifier)")
print("=" * 50)
for k in [1, 3, 5, 7, 9, 11]:
    clf_k = KNeighborsClassifier(n_neighbors=k)
    clf_k.fit(X_train, y_train_cls)
    acc = accuracy_score(y_test_cls, clf_k.predict(X_test))
    print(f"  k={k:2d}  ->  Accuracy: {acc:.4f}")


Accuracy vs k (KNeighborsClassifier)
  k= 1  ->  Accuracy: 0.9667
  k= 3  ->  Accuracy: 0.9333
  k= 5  ->  Accuracy: 0.9333
  k= 7  ->  Accuracy: 0.9667
  k= 9  ->  Accuracy: 0.9667
  k=11  ->  Accuracy: 0.9667
